# Preference-model tables and plots

Assemble publication-ready CSV/LaTeX tables and vector figures from exported regression, XGBoost, and optional text-augmented neural artifacts. Every ranker uses the shared feature and sealed-test contract. This notebook does not inspect model internals or recompute fitted models.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from commentgap_analysis.reporting import build_reporting_outputs
from commentgap_analysis.neural_reporting import build_neural_reporting_outputs

FEATURE_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
REGRESSION_ROOT = Path(os.getenv("COMMENTGAP_REGRESSION_ROOT", "model_output/selection_2025/regression"))
XGB_ROOT = Path(os.getenv("COMMENTGAP_XGB_ROOT", "model_output/selection_2025/xgboost_paper2"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_REPORT_ROOT", "model_output/selection_2025/reporting_paper2"))
INCLUDE_NEURAL = os.getenv("COMMENTGAP_INCLUDE_NEURAL", "0").lower() in {"1", "true", "yes"}
NEURAL_ROOTS = {
    "Frozen BGE-M3": Path(os.getenv("COMMENTGAP_FROZEN_BGE_ROOT", "model_output/selection_2025/neural_rankers/frozen_bge_m3")),
    "Metadata-only MLP": Path(os.getenv("COMMENTGAP_METADATA_MLP_ROOT", "model_output/selection_2025/neural_rankers/metadata_mlp")),
}
provenance = json.loads((FEATURE_ROOT / "provenance_manifest.json").read_text())
provenance["watermark"]

## Outputs

Generate the existing regression/XGBoost report, then optionally add sealed-test neural performance, paired XGBoost comparisons, tie sensitivity, and a combined SVG/PDF figure.

In [ ]:
report = build_reporting_outputs(
    FEATURE_ROOT,
    REGRESSION_ROOT,
    XGB_ROOT,
    OUTPUT_ROOT,
)
neural_report = build_neural_reporting_outputs(
    XGB_ROOT, NEURAL_ROOTS, OUTPUT_ROOT,
) if INCLUDE_NEURAL else {"models": [], "tables": [], "figures": []}
{"base": report, "neural": neural_report}

## Sealed-test model comparison

Display the original article-bootstrap comparison and, when enabled, all model summaries plus paired neural-minus-XGBoost differences. Intervals crossing zero do not establish a performance difference.

In [ ]:
model_test_performance = pd.read_csv(OUTPUT_ROOT / "tables/model_test_performance.csv")
paired_test_differences = pd.read_csv(OUTPUT_ROOT / "tables/regression_vs_xgb_test_differences.csv")
display(model_test_performance)
display(paired_test_differences)
if INCLUDE_NEURAL:
    all_model_test_performance = pd.read_csv(OUTPUT_ROOT / "tables/all_model_test_performance.csv")
    xgb_vs_neural = pd.read_csv(OUTPUT_ROOT / "tables/xgb_vs_neural_test_differences.csv")
    display(all_model_test_performance)
    display(xgb_vs_neural)

## Regression robustness outputs

Display conditional-logit fit diagnostics and frozen-score test sensitivity to valid audience tie resolutions. The probability-contrast table remains a heuristic logistic translation, not an exact fixed-k inclusion-probability calculation.

In [ ]:
regression_diagnostics = pd.read_csv(OUTPUT_ROOT / "tables/regression_model_diagnostics.csv")
regression_above_chance = pd.read_csv(OUTPUT_ROOT / "tables/regression_test_above_chance.csv")
test_tie_sensitivity = pd.read_csv(OUTPUT_ROOT / "tables/model_test_tie_sensitivity.csv")
display(regression_diagnostics)
display(regression_above_chance)
display(test_tie_sensitivity)

## Acceptance checks

The report manifest inherits the feature watermark. Pilot figures and tables must remain visibly separated from inference-ready outputs.

In [ ]:
manifest = json.loads((OUTPUT_ROOT / "report_manifest.json").read_text())
assert manifest["watermark"] == provenance["watermark"]
assert manifest["tables"] and manifest["figures"]
required_tables = {
    "model_test_performance.csv",
    "regression_vs_xgb_test_differences.csv",
    "regression_model_diagnostics.csv",
    "regression_test_chance_baseline.csv",
    "regression_test_above_chance.csv",
    "model_test_tie_sensitivity.csv",
}
assert required_tables.issubset(manifest["tables"])
if INCLUDE_NEURAL:
    required_neural = {
        "all_model_test_performance.csv",
        "xgb_vs_neural_test_differences.csv",
        "all_model_test_tie_sensitivity.csv",
    }
    assert all((OUTPUT_ROOT / "tables" / name).exists() for name in required_neural)
manifest

## Interpretation note

Regression panels compare curator and audience associations on the shared stacked-selection scale. Conditional logit, XGBoost, and neural rankers use the same sealed 50% Paper 2 articles and are compared with paired article-level differences. None should be described as causal evidence or as a live-ranking deployment evaluation.